In [7]:
from __future__ import annotations
 
import numpy as np
import pandas as pd
from numpy.random import Generator, RandomState
 
# --- arch imports ---
from arch.univariate import ZeroMean, ConstantMean, ARX, GARCH, ConstantVariance
from arch.univariate.distribution import (
    Distribution,
    Normal,
    StudentsT,
    SkewStudent,
    GeneralizedError,
)

In [2]:
n=1000
mu=0.15
omega = 0.05
alpha = 0.05
beta = 0.90
distribution = None
seed=42

In [ ]:
import numpy as np
from arch import arch_model
from arch.univariate import StudentsT

# 1. Crear una clase personalizada heredando de StudentsT
class CustomStudentsT(StudentsT):
    def constraints(self):
        # Mantenemos la matriz A igual, pero cambiamos el vector b
        # A * theta >= b  --->  nu >= 4.01
        return np.array([[1], [-1]]), np.array([4.01, -500.0])

    def bounds(self, resids):
        # Actualizamos la tupla de límites para el optimizador (L-BFGS-B / SLSQP)
        return [(4.05, 500.0)]


In [40]:
m = ConstantMean()
m.volatility = ConstantVariance()
seed=42
m.distribution = StudentsT(seed=seed)
params = [0.15, 0.3] + [8.1]

result = m.simulate(params, nobs=10000, burn=100)


In [41]:
from arch import arch_model

am = arch_model(result.data, mean='Constant', vol='constant', dist='t', rescale=False)
am.distribution = CustomStudentsT()

In [42]:
res_fit = am.fit()

Iteration:      1,   Func. Count:      5,   Neg. LLF: 533390.6830810492
Iteration:      2,   Func. Count:     13,   Neg. LLF: 284498.6686737074
Iteration:      3,   Func. Count:     19,   Neg. LLF: 8088.256440957029
Iteration:      4,   Func. Count:     24,   Neg. LLF: 7968.759066596889
Iteration:      5,   Func. Count:     28,   Neg. LLF: 7968.700501688469
Iteration:      6,   Func. Count:     32,   Neg. LLF: 7968.697715893464
Iteration:      7,   Func. Count:     36,   Neg. LLF: 7968.697699686558
Iteration:      8,   Func. Count:     39,   Neg. LLF: 7968.697699686562
Optimization terminated successfully    (Exit mode 0)
            Current function value: 7968.697699686558
            Iterations: 8
            Function evaluations: 39
            Gradient evaluations: 8


In [43]:
res_fit.params

mu        0.152707
sigma2    0.296259
nu        8.364622
Name: params, dtype: float64

In [44]:
6/(res_fit.params['nu']-4)

np.float64(1.3746895800450145)

In [45]:

from scipy import stats

stats.kurtosis(result.data, fisher=True)

np.float64(1.083914882017475)

In [ ]:
res_fit.params

In [1]:
from core.dgp import GARCHProcess
import numpy as np
from scipy import stats

In [2]:
dgp = GARCHProcess(mu=0.05, omega=0.05, alpha=0.10, beta=0.85,
             dist='t', dist_params=[8, 0]).calibrate_params(0.15, 0.3)

In [3]:
rng = np.random.default_rng(42)
res = dgp.simulate(1000, rng)

In [12]:
res.mean(),res.std()

(np.float64(0.1440580765323071), np.float64(0.2908905778707369))

In [4]:
from arch import arch_model

am = arch_model(res, mean='Constant', vol='GARCH',p=1,q=1, dist='t', rescale=False)

In [5]:
res_fit = am.fit(update_freq=0,disp=False)

In [6]:
res_fit.params

mu          0.144213
omega       0.007142
alpha[1]    0.103805
beta[1]     0.804886
nu          6.815052
Name: params, dtype: float64

In [22]:
res_fit.params.tolist()

[1.458919806004956,
 1.4223865299041283,
 0.13963322357956667,
 0.6947177032745095]

In [23]:
res_fit.params['alpha[1]']

np.float64(0.13963322357956667)

In [24]:
from core.models import GARCH11Model

model = GARCH11Model()

In [25]:
model.fit(res)

{'omega': np.float64(1.4223865299041283),
 'alpha': np.float64(0.13963322357956667),
 'beta': np.float64(0.6947177032745095),
 'skew': 0.06744140746491294,
 'exc_kurt': 0.44746189485031795}

In [1]:
import numpy as np

rng = np.random.default_rng(42)

# Simulate three series with different DGPs
n1, n2, n3 = 500, 300, 700

# Series 0: pure iid normal
s0 = rng.normal(0.05, 1.0, n1)

In [3]:
from arch.univariate import (
    ARX,
    ConstantMean,
    ConstantVariance,
    GARCH,
    GeneralizedError,
    Normal,
    SkewStudent,
    StudentsT,
)

In [ ]:
x = s0
m = ConstantMean(x)
m.volatility = ConstantVariance()
m.distribution = Normal()
model = m
res = model.fit(disp="off", show_warning=False)
print(res.summary())

               Constant Mean - Constant Variance Model Results                
Dep. Variable:                      y   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:          Constant Variance   Log-Likelihood:               -688.523
Distribution:                  Normal   AIC:                           1381.05
Method:            Maximum Likelihood   BIC:                           1389.48
                                        No. Observations:                  500
Date:                Sat, Mar 28 2026   Df Residuals:                      499
Time:                        11:17:51   Df Model:                            1
                                 Mean Model                                
                 coef    std err          t      P>|t|     95.0% Conf. Int.
---------------------------------------------------------------------------
mu             0.0369  4.289e-02      0.860      0.390 [-4.71

In [5]:
x = s0
m = ARX(x, lags=1)
m.volatility = ConstantVariance()
m.distribution = Normal()
model = m
res = model.fit(disp="off", show_warning=False)
print(res.summary())

                     AR - Constant Variance Model Results                     
Dep. Variable:                      y   R-squared:                       0.010
Mean Model:                        AR   Adj. R-squared:                  0.008
Vol Model:          Constant Variance   Log-Likelihood:               -685.114
Distribution:                  Normal   AIC:                           1376.23
Method:            Maximum Likelihood   BIC:                           1388.87
                                        No. Observations:                  499
Date:                Sat, Mar 28 2026   Df Residuals:                      497
Time:                        11:16:49   Df Model:                            2
                                 Mean Model                                
                 coef    std err          t      P>|t|     95.0% Conf. Int.
---------------------------------------------------------------------------
Const          0.0323  4.259e-02      0.758      0.449 [-5.12